In [ ]:

import theano
print('theano: %s' % theano.__version__) 

In [ ]:
# keras 
import keras 
print('keras: %s' % keras.__version__) 

In [ ]:
#pydot should be installed via Anaconda prompt:
#conda install pydot
#conda install graphviz
#conda install python-graphviz
#install graphviz from here http://www.graphviz.org/download/

import pandas as pd
import numpy as np
from keras.models import Sequential 
from keras.layers import Dense,Activation
from keras import backend as K
#import needed to create custom activation function
from keras.utils.generic_utils import get_custom_objects
from keras.utils.vis_utils import plot_model
import pydot

#we want repeatable results, so we fix random seed,seed is used for random weight initialisation 
np.random.seed(10) 
 
#download data from:https://archive.ics.uci.edu/ml/datasets/Haberman's+Survival
# Attribute Information:
#   1. Age of patient at time of operation (numerical)
#   2. Patient's year of operation (year - 1900, numerical)
#   3. Number of positive axillary nodes detected (numerical)
#   4. Survival status (class attribute)
#         1 = the patient survived 5 years or longer
#         2 = the patient died within 5 year


 
# load data from data.csv file into ds - numpy array
df = pd.read_csv('haberman.csv')

#prepare labels in y column,change values into -1 or 1
df.loc[df['y'] == 1, 'y'] = -1
df.loc[df['y'] == 2, 'y'] = 1
print(df.head(10))

In [ ]:
X = df.iloc[:,0:3] 
Y = df.iloc[:,3] 

# normalize X data
X=(X-X.mean())/X.std()
X.head()

In [ ]:
#Split data into TrainingSet[input (X) and output (Y) variables]  
# and TestingSet[input (X2) and output (Y2) variables] 

#Calculate number of rows in the training set
trainingSetSize=int(df.shape[0]*0.8) 
print("Number of rows in training set",trainingSetSize)

X1 = X.iloc[:trainingSetSize,:] 
Y1 = Y.iloc[:trainingSetSize] 

X2 = X.iloc[trainingSetSize: ,:] 
Y2= Y.iloc[trainingSetSize:]

In [ ]:
import pydot
import graphviz
import os
from IPython.display import display, Image
from matplotlib import pyplot
from keras import backend
from keras.layers import Dense, Dropout, Flatten, Activation, BatchNormalization

def rmse(y_true, y_pred):
    return backend.sqrt(backend.mean(backend.square(y_pred - y_true), axis=-1))

#add C:\Program Files (x86)\Graphviz2.38\bin to PATH variable
#os.environ["PATH"] += os.pathsep + 'C:/Program Files (x86)/Graphviz2.38/bin/'

#Create nn model
#use Sequential class which is a linear stack of Layers
nn_model = Sequential() 

#Define the structure of each layer in nn
#For each layer set info about: 
#number of neurons
#activation function type:linear, relu,sigmoid, tanh

#Input layer (3 neurons) + 1st hidden layer 
nn_model.add(Dense(4, input_dim=3, activation='relu')) 
nn_model.add(BatchNormalization())

#2nd hidden layaer  
nn_model.add(Dense(3, activation='tanh')) 
nn_model.add(BatchNormalization())

#Output layaer 
nn_model.add(Dense(1,activation='tanh'))

#Compile model

#optional values of loss param: mse,binary_crossentropy,categorical_crossentropy,more about loss function types: https://keras.io/losses/
#use Adam optmizer with default values:Adam(lr=0.001, beta_1=0.9, beta_2=0.999, epsilon=None, decay=0.0, amsgrad=False)

#metrics - function, which describes how do we measure similiarity between the value predicted by mode, outputs are stored 
#in Y vector

#rmse - custom metric defined above
nn_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

#Plot structure of the nn model
plot_model(nn_model,to_file='model_plot.png', show_shapes=True, show_layer_names=True)
i=Image("model_plot.png")
display(i)

In [ ]:
from scipy import stats

#Train the nn model-training a network is finding parameters that minimize a loss function (or cost function).
#epochs:number of iteration in training procedure
#batch_size:number of observation in each itearation, after which the weights are updated
#verbose:set progress bar visibility, 0 = silent, 1 = progress bar, 2 = one line per epoch
history=nn_model.fit(X1.as_matrix(), Y1.as_matrix(), epochs=450, batch_size=10,verbose=0) 

#Evaluate the model on the training set
results_train = nn_model.evaluate(X1.as_matrix(), Y1.as_matrix()) 
print('Loss function value - training set:', results_train[0])
print('Metrics value -  training set:',  results_train[1])

#Predict values in the training set
Y1_pred=nn_model.predict(X1)

#Assign category -1 or 1 basing on the predicted values - training set  
Y1_pred[Y1_pred>=0]=1
Y1_pred[Y1_pred<0]=-1

print("Success rate(%) - training set:",100*np.sum(Y1_pred[:,0]==Y1.as_matrix())/len(Y1))
print("Number of obs. per category in Y1 - training_set")

print(stats.itemfreq(Y1.as_matrix()))

In [ ]:
#Make predictions on the testing set
#Y2_pred is numpy array
Y2_pred= nn_model.predict(X2)

#Assign category -1 or 1 basing on the predicted values - testing set  
Y2_pred[Y2_pred>=0]=1
Y2_pred[Y2_pred<0]=-1

print("Success rate(%) - testing set:",100*np.sum(Y2_pred[:,0]==Y2.as_matrix())/len(Y2))
print("Number of obs. per category in Y2 - testing set")

#print(stats.itemfreq(Y2.as_matrix()))
print("\n\n")